# Red neuronal para clasificación de MNIST con TensorFlow/Keras

## Objetivo
Construir una red neuronal para reconocer dígitos manuscritos de MNIST de forma **modular y didáctica**.

La idea es separar claramente cada etapa del proceso:

1. Cargar los datos.
2. Inspeccionar las dimensiones y las etiquetas.
3. Visualizar imágenes de entrenamiento.
4. Separar los datos de entrenamiento en **train** y **validación**.
5. Normalizar los datos.
6. Transformar cada imagen de 28×28 en un vector de 784 elementos.
7. Construir la arquitectura de la red neuronal.
8. Configurar el optimizador y la función de pérdida.
9. Entrenar usando entrenamiento + validación.
10. Analizar gráficamente pérdida y exactitud.
11. Evaluar finalmente con el conjunto de prueba (**test**), que no participa en el entrenamiento.

> **Idea académica importante:** MNIST proporciona inicialmente `train` y `test`. Para estudiar correctamente el entrenamiento y la validación, dividiremos el conjunto `train` en dos: `train` y `validation`. El conjunto `test` se mantiene separado hasta la evaluación final.

## Módulo 1 — Importar librerías

En este módulo cargamos las herramientas necesarias para trabajar con MNIST, visualizar imágenes y construir la red neuronal.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

print("TensorFlow:", tf.__version__)

## Módulo 2 — Cargar MNIST

MNIST contiene imágenes de dígitos manuscritos de tamaño **28×28 píxeles**.

- `X_train`: imágenes para desarrollar el modelo.
- `y_train`: etiquetas correspondientes a esas imágenes.
- `X_test`: imágenes reservadas para la evaluación final.
- `y_test`: etiquetas del conjunto de prueba.

Cada etiqueta es un número entre 0 y 9.

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

### ¿Qué significan estas dimensiones?

Esperamos obtener:

- `X_train = (60000, 28, 28)` → 60 000 imágenes de 28×28.
- `y_train = (60000,)` → una etiqueta por imagen.
- `X_test = (10000, 28, 28)` → 10 000 imágenes para prueba.
- `y_test = (10000,)` → una etiqueta por imagen de prueba.

Todavía no hemos convertido las imágenes en vectores. En este punto conservamos la estructura original de 28×28.

## Módulo 3 — Explorar y visualizar los datos de entrenamiento

Antes de entrenar una red es conveniente observar físicamente los datos que estamos utilizando.

El siguiente bloque permite seleccionar una imagen mediante un campo numérico.

Al pulsar **Mostrar imagen** veremos, en este orden:

1. La imagen del dígito.
2. El tamaño de la imagen: `28 × 28`.
3. La etiqueta correspondiente (`0` a `9`).
4. Los valores mínimo y máximo de los píxeles.
5. **La matriz 28 × 28 de datos que representa la imagen.**

### ¿Qué estamos viendo?

La imagen no es almacenada por la red como un dibujo. Es una **matriz numérica de 28 × 28**. Cada posición de esa matriz corresponde a un píxel.

En este módulo todavía observamos los datos originales de MNIST, cuyos valores de píxel están entre **0 y 255**. Más adelante, en el módulo de normalización, transformaremos estos valores al intervalo **0–1** dividiendo entre 255.


In [ ]:
# Campo para seleccionar una imagen del conjunto de entrenamiento
indice = widgets.IntText(
    value=0,
    description='Imagen:',
    min=0,
    max=len(X_train)-1
)

boton = widgets.Button(description='Mostrar imagen')
salida = widgets.Output()

def mostrar_imagen(b):
    with salida:
        clear_output(wait=True)
        i = indice.value

        # 1. Mostrar la imagen
        plt.figure(figsize=(4, 4))
        plt.imshow(X_train[i], cmap='gray')
        plt.title(f'Imagen {i} — Dígito: {y_train[i]}')
        plt.axis('off')
        plt.show()

        # 2. Mostrar información básica
        print('Tamaño de la imagen:', X_train[i].shape)
        print('Etiqueta:', y_train[i])
        print('Valor mínimo del píxel:', X_train[i].min())
        print('Valor máximo del píxel:', X_train[i].max())

        # 3. Mostrar debajo de la imagen los datos de los píxeles
        print('\nMatriz de datos de la imagen (28 × 28 píxeles):')
        print(X_train[i])

boton.on_click(mostrar_imagen)

display(indice, boton, salida)


## Módulo 4 — Separar entrenamiento y validación

MNIST entrega 60 000 ejemplos de entrenamiento. Vamos a dividirlos en:

- **80 % → entrenamiento**: la red aprende sus parámetros con estos datos.
- **20 % → validación**: permite observar cómo se comporta la red frente a datos que no utilizó para actualizar sus pesos.

El conjunto `test` de 10 000 imágenes permanece completamente separado.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("Después de la separación:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val  :", X_val.shape)
print("y_val  :", y_val.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

### Interpretación

Ahora tenemos tres grupos conceptualmente diferentes:

`TRAIN` → aprende los pesos de la red.

`VALIDATION` → controla el comportamiento durante el entrenamiento y ayuda a detectar sobreajuste.

`TEST` → se reserva para una evaluación final, después de haber tomado las decisiones sobre el modelo.

Esta separación es fundamental para evitar confundir validación con evaluación final.

## Módulo 5 — Normalización de las imágenes

Los píxeles originales de MNIST están en el intervalo **0–255**.

Los convertimos al intervalo **0–1** mediante:

$$
x_{normalizado}=\frac{x}{255}
$$

Esto facilita el entrenamiento de la red porque trabajamos con valores de menor magnitud.

In [ ]:
X_train = X_train.astype("float32") / 255.0
X_val   = X_val.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

print("Mínimo X_train:", X_train.min())
print("Máximo X_train:", X_train.max())
print("Mínimo X_val  :", X_val.min())
print("Máximo X_val  :", X_val.max())
print("Mínimo X_test :", X_test.min())
print("Máximo X_test :", X_test.max())

## Módulo 6 — Convertir cada imagen 28×28 en un vector de 784 elementos

La arquitectura que vamos a utilizar es una red **fully connected (Dense)**. Por ello, cada imagen debe convertirse en un vector.

Como:

$$
28\times28=784
$$

cada imagen pasará de:

`28 × 28` → `784`

La matriz resultante tendrá una fila por imagen.

In [ ]:
X_train_flat = X_train.reshape(X_train.shape[0], 784)
X_val_flat   = X_val.reshape(X_val.shape[0], 784)
X_test_flat  = X_test.reshape(X_test.shape[0], 784)

print("Matriz de entrenamiento:", X_train_flat.shape)
print("Matriz de validación   :", X_val_flat.shape)
print("Matriz de prueba       :", X_test_flat.shape)

### ¿Qué representa cada matriz?

| Matriz | Filas | Columnas | Uso |
|---|---:|---:|---|
| `X_train_flat` | 48 000 | 784 | Entrenamiento |
| `X_val_flat` | 12 000 | 784 | Validación |
| `X_test_flat` | 10 000 | 784 | Evaluación final |

Las etiquetas permanecen separadas:

- `y_train` → etiquetas de entrenamiento.
- `y_val` → etiquetas de validación.
- `y_test` → etiquetas de prueba.

## Módulo 7 — Construir la arquitectura de la red

Vamos a comenzar con una arquitectura similar a la mostrada en las diapositivas:

- **Entrada:** 784 neuronas.
- **Primera capa oculta:** 512 neuronas + ReLU.
- **Segunda capa oculta:** 512 neuronas + ReLU.
- **Salida:** 10 neuronas + Softmax.

La salida tiene 10 neuronas porque existen 10 clases: 0, 1, 2, ..., 9.

In [ ]:
model = Sequential(name="MNIST_Dense")

model.add(Dense(512, activation='relu', input_shape=(784,), name='hidden_1'))
model.add(Dense(512, activation='relu', name='hidden_2'))
model.add(Dense(10, activation='softmax', name='output'))

model.summary()

## Módulo 8 — ¿Cuántos parámetros tiene la red?

En una capa Dense, cada conexión tiene un peso y cada neurona tiene un sesgo.

Para la primera capa:

$$
784\times512+512=401\,920
$$

Para la segunda:

$$
512\times512+512=262\,656
$$

Para la salida:

$$
512\times10+10=5\,130
$$

Estos son los parámetros que la red aprenderá durante el entrenamiento.

## Módulo 9 — Configurar el entrenamiento

Utilizaremos:

- **Función de pérdida:** `sparse_categorical_crossentropy`, apropiada porque las etiquetas son enteros de 0 a 9.
- **Optimizador:** Adam.
- **Métrica:** accuracy.

La función de pérdida indica qué tan equivocada está la red y Adam ajusta los pesos para reducir esa pérdida.

In [ ]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print("Modelo compilado correctamente.")

## Módulo 10 — Entrenar la red

Aquí ocurre el aprendizaje.

En cada época la red recibe ejemplos de `X_train_flat` junto con sus etiquetas `y_train`. Al finalizar cada época, Keras evalúa el modelo sobre `X_val_flat` y `y_val` para calcular las métricas de validación.

La variable `history` guarda la evolución de pérdida y accuracy.

In [ ]:
history = model.fit(
    X_train_flat,
    y_train,
    epochs=20,
    batch_size=128,
    validation_data=(X_val_flat, y_val),
    verbose=1
)

## Módulo 11 — Graficar pérdida: entrenamiento vs validación

La primera gráfica permite estudiar la evolución del **loss**.

### ¿Qué buscamos?

- Si entrenamiento y validación disminuyen de manera similar → comportamiento saludable.
- Si el loss de entrenamiento sigue disminuyendo pero el de validación comienza a aumentar → posible **overfitting (sobreajuste)**.
- Si ambos permanecen altos → puede existir **underfitting (subajuste)** o un problema con la arquitectura/entrenamiento.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.title('Pérdida durante el entrenamiento')
plt.legend()
plt.grid(True)
plt.show()

## Módulo 12 — Graficar accuracy: entrenamiento vs validación

Ahora observamos el porcentaje de clasificaciones correctas.

La comparación entre ambas curvas permite estudiar la capacidad de generalización del modelo.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.title('Exactitud durante el entrenamiento')
plt.legend()
plt.grid(True)
plt.show()

## Módulo 13 — Evaluación sobre validación

Aunque las curvas ya muestran el comportamiento, podemos obtener explícitamente las métricas del conjunto de validación.

In [ ]:
val_loss, val_accuracy = model.evaluate(
    X_val_flat,
    y_val,
    verbose=0
)

print(f"Validation loss    : {val_loss:.6f}")
print(f"Validation accuracy: {val_accuracy:.6f}")

## Módulo 14 — Evaluación final sobre TEST

Este conjunto no fue utilizado para ajustar los pesos ni para seleccionar la arquitectura.

Por eso representa mejor la capacidad de generalización final del modelo.

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test_flat,
    y_test,
    verbose=0
)

print(f"Test loss    : {test_loss:.6f}")
print(f"Test accuracy: {test_accuracy:.6f}")

## Módulo 15 — Realizar predicciones

La red produce 10 probabilidades, una para cada dígito. La clase con mayor probabilidad se toma como predicción.

In [ ]:
probabilidades = model.predict(X_test_flat[:10], verbose=0)
predicciones = np.argmax(probabilidades, axis=1)

print("Predicciones:", predicciones)
print("Etiquetas reales:", y_test[:10])

## Módulo 16 — Visualizar predicciones

Comparamos visualmente la imagen, la etiqueta real y la predicción de la red.

In [ ]:
plt.figure(figsize=(12, 5))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[i], cmap='gray')
    plt.title(f'Real: {y_test[i]} | Pred: {predicciones[i]}')
    plt.axis('off')

plt.tight_layout()
plt.show()

## Módulo 17 — Resumen conceptual del flujo

El flujo completo que acabamos de implementar es:

```text
MNIST
  │
  ├── TRAIN original (60 000)
  │       │
  │       ├── TRAIN (48 000) ──→ normalizar ──→ 784 ──→ red ──→ pesos
  │       │
  │       └── VALIDATION (12 000) ──→ normalizar ──→ 784 ──→ evaluar durante entrenamiento
  │
  └── TEST (10 000) ──→ normalizar ──→ 784 ──→ evaluación final
```

### En términos de matrices

$$
X_{train}\in\mathbb{R}^{48000\times784}
$$

$$
X_{val}\in\mathbb{R}^{12000\times784}
$$

$$
X_{test}\in\mathbb{R}^{10000\times784}
$$

y cada fila de `X` tiene una etiqueta asociada en `y`.

# Módulo opcional — Agregar Dropout para reducir overfitting

Una vez entendido el modelo base, podemos construir una segunda versión agregando `Dropout`.

Por ejemplo, con `rate=0.4`, durante el entrenamiento se desactivan aleatoriamente aproximadamente el 40 % de las activaciones de esa capa en cada actualización.

Esto obliga a la red a no depender demasiado de neuronas individuales y puede mejorar la generalización.

> **Importante:** Dropout se activa durante entrenamiento. Durante validación y test Keras lo maneja automáticamente en modo de evaluación.

In [ ]:
from tensorflow.keras.layers import Dropout

model_dropout = Sequential(name="MNIST_Dense_Dropout")
model_dropout.add(Dense(512, activation='relu', input_shape=(784,), name='hidden_1'))
model_dropout.add(Dropout(0.4, name='dropout_1'))
model_dropout.add(Dense(512, activation='relu', name='hidden_2'))
model_dropout.add(Dropout(0.4, name='dropout_2'))
model_dropout.add(Dense(10, activation='softmax', name='output'))

model_dropout.summary()

In [ ]:
model_dropout.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history_dropout = model_dropout.fit(
    X_train_flat,
    y_train,
    epochs=20,
    batch_size=128,
    validation_data=(X_val_flat, y_val),
    verbose=1
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_dropout.history['loss'], label='Entrenamiento')
plt.plot(history_dropout.history['val_loss'], label='Validación')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.title('Modelo con Dropout — pérdida')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_dropout.history['accuracy'], label='Entrenamiento')
plt.plot(history_dropout.history['val_accuracy'], label='Validación')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.title('Modelo con Dropout — exactitud')
plt.legend()
plt.grid(True)
plt.show()

# Conclusión

El notebook está organizado para que cada bloque pueda explicarse independientemente en una clase o presentación.

La secuencia metodológica es:

**Datos → inspección → visualización → separación train/validation → normalización → vectorización 784 → arquitectura → compilación → entrenamiento → validación → gráficas → test → predicción.**

La versión con Dropout queda como extensión natural para estudiar el **overfitting y las técnicas de regularización**.